In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import PolynomialFeatures
import gc




In [2]:
dtype_dict = {
    "R": "int8",
    "C": "int8",
    "breath_id": "int32",
    "id": "int16",
    "time_step": "float32",
    "u_in": "float32",
    "u_out": "int8",
    "pressure": "float32",
}
df_train = pd.read_csv(
    "../input/ventilator-pressure-prediction/train.csv",
    dtype=dtype_dict,
    usecols=["R", "C", "breath_id", "id", "time_step", "u_in", "u_out", "pressure"],
)
df_test = pd.read_csv(
    "../input/ventilator-pressure-prediction/test.csv",
    dtype=dtype_dict,
    usecols=["R", "C", "breath_id", "id", "time_step", "u_in", "u_out"],
)




In [3]:
df_train["cum_u_in"] = df_train.groupby("breath_id", sort=False)["u_in"].cumsum()
df_test["cum_u_in"] = df_test.groupby("breath_id", sort=False)["u_in"].cumsum()
df_train["delta_u_in"] = (
    df_train.groupby("breath_id", sort=False)["u_in"].diff().fillna(0)
)
df_test["delta_u_in"] = (
    df_test.groupby("breath_id", sort=False)["u_in"].diff().fillna(0)
)
df_train["delta_time"] = (
    df_train.groupby("breath_id", sort=False)["time_step"].diff().fillna(0)
)
df_test["delta_time"] = (
    df_test.groupby("breath_id", sort=False)["time_step"].diff().fillna(0)
)

feature_base = [
    "R",
    "C",
    "u_in",
    "u_out",
    "time_step",
    "cum_u_in",
    "delta_u_in",
    "delta_time",
]

df_train["RC"] = df_train["R"].astype(np.int16) * df_train["C"].astype(np.int16)
df_test["RC"] = df_test["R"].astype(np.int16) * df_test["C"].astype(np.int16)
feature_base.append("RC")

poly = PolynomialFeatures(degree=2, include_bias=False)

base_train = df_train[feature_base].astype(np.float32).values
base_test = df_test[feature_base].astype(np.float32).values

X_train_poly = poly.fit_transform(base_train).astype(np.float32, copy=False)
X_test_poly = poly.transform(base_test).astype(np.float32, copy=False)

test_ids = df_test["id"].astype(np.int16).values

id_features_train = df_train[["breath_id", "id"]].astype(np.float32).values
X_train = np.hstack([X_train_poly, id_features_train])

id_features_test = df_test[["breath_id", "id"]].astype(np.float32).values
X_test = np.hstack([X_test_poly, id_features_test])

y_train = df_train["pressure"].astype(np.float32).values

del (
    df_train,
    df_test,
    base_train,
    base_test,
    X_train_poly,
    X_test_poly,
    id_features_train,
    id_features_test,
)
gc.collect()




0

In [4]:
# Reduced max_iter to cut runtime; early‑stopping (n_iter_no_change) will still stop earlier if needed.
model = HistGradientBoostingRegressor(
    max_iter=800,
    learning_rate=0.01,
    max_depth=7,
    random_state=42,
    loss="absolute_error",
    verbose=0,
    max_bins=63,
    n_iter_no_change=10,
    tol=1e-5,
)




In [5]:
model.fit(X_train, y_train)
test_pred = model.predict(X_test)




In [6]:
df_sub = pd.DataFrame({"id": test_ids, "pressure": test_pred})
df_sub.to_csv("submission.csv", index=False)




In [7]:
print(df_sub.head())

   id   pressure
0   1   6.064487
1   2   5.897505
2   3   6.096011
3   4   9.821444
4   5  13.168266
